IA & Data science (LU3IN0226) -- 2025-2026
--------
*&copy; Equipe pédagogique: Christophe Marsala, Olivier Schwander, Jean-Noël Vittaut, Maxellende Julienne.*


# TD-TME 6 : Apprentissage pour le texte

<font size="+1" color="RED"><b>[Q]</b></font> **Indiquer dans la boîte ci-dessous vos noms et prénoms :**

*Double-cliquer ici et insérer les noms et prénoms de votre binôme*

<font color="RED" size="+1"><b>[Q]</b></font> **Renommer ce notebook**

Tout en haut de cette page, cliquer sur <tt>tme-06</tt> et rajouter à la suite de <tt>tme-06</tt> les noms des membres du binômes séparés par un tiret.

<font color="RED" size="+1">IMPORTANT: soumission de votre fichier final</font>

**Nom à donner au fichier à poster** : *tme-06-Nom1_Nom2.ipynb* 
- *Nom1* et *Nom2* : noms des membres du binôme
- ne pas compresser ou faire une archive: il faut rendre le fichier ipython tel quel, éventuellement, si vous avez d'autres fichiers vous les rendez séparément.

**Echancier pour la soumission de votre compte-rendu:**
- le compte-rendu d'une séance doit être remis obligatoirement <font color="RED">avant la séance suivante</font>.

**Le compte-rendu est soumis sur la page Moodle.**

In [6]:
# - - - - - - - - - - - - - - - - - -
# imports utiles
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mtpl
%matplotlib inline  

import math
import time
import sys

# Les instructions suivantes sont utiles pour recharger automatiquement 
# le code modifié dans les librairies externes
%load_ext autoreload
%autoreload 2

# - - - - - - - - - - - - - - - - - -
# Information sur l'environnent utilisé ici:
print("Version python et des librairies:")
print("\tPython ",sys.version)
print("\tpandas: ",pd.__version__)
print("\tnumpy: ",np.__version__)
print("\tmatplotlib: ",mtpl.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Version python et des librairies:
	Python  3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]
	pandas:  3.0.2
	numpy:  2.4.4
	matplotlib:  3.10.8


In [7]:
# Importation de votre librairie iads:
# La ligne suivante permet de préciser le chemin d'accès à la librairie iads
import sys
sys.path.append('../')   # iads doit être dans le répertoire père du répertoire courant !

# Importation de la librairie iads
import iads as iads

# importation de Classifiers
from iads import Classifiers as cl

# importation de utils
from iads import utils as ut

# importation de evaluation
from iads import evaluation as ev



# Objectifs de ce TME

<div class="alert alert-block alert-warning">
Ce TME a pour but d'implémenter des fonctions pour traiter un corpus textuel et un nouvel algorithme d'apprentissage (reportez-vous aux supports du cours 6). 

Pour expérimenter, on utilise la base `SMS spam Collection` qui contient 5572 messages associés à 2 labels: `spam` et `ham`. 
</div>

In [8]:
# Chargement du dataset

df_spam = pd.read_csv('data/spam.csv', sep='\t', encoding = 'latin1')
df_spam

,label,message
0,ham,Go until jurong point crazy.. Available only ...
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,Nah I don't think he goes to usf he lives aro...
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,Pity * was in mood for that. So...any other s...
5570,ham,The guy did some bitching but I acted like i'd...


In [9]:
# Valeurs possibles pour le label:
df_spam['label'].unique()

<StringArray>
['ham', 'spam']
Length: 2, dtype: str

In [10]:
# On met les labels dans une liste car cela sera utile:
les_labels = df_spam['label'].unique()

<font color="RED" size="+1"><b>[Q]</b></font> En utilisant `value_counts` (voir la doc de la librairie pandas), afficher le nombre d'exemples de chaque classe dans la base.

In [ ]:
# calcule cb docs on a de chaque label
print(df_spam['label'].value_counts())


label
ham     4825
spam     747
Name: count, dtype: int64


## Prétraitements

### Nettoyage des données

Pour pouvoir travailler sur les messages, on commence par les nettoyer : il faut enlever les mots inutiles (caractères de ponctuations, les articles, les mots trop courants, etc.) (voir le cours 6).

Pour traiter les données textuelles (colonne `message` du dataset), on utilise les fonctions de la librairie `string`.

In [38]:
import string

print("Caractères de ponctuations : ", string.punctuation)

print("Mise en minuscules (pour homogénéiser l'écriture : ", "May the Force be with you!".lower())

print("Découper une phrase avec espace (retourne une liste):", "LU3IN026 est l'UE d'IA et Sciences des données.".split())

print("Découper une phrase avec apostrophe (retourne une liste):", "LU3IN026 est l'UE d'IA et Sciences des données.".split("'"))



Caractères de ponctuations :  !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Mise en minuscules (pour homogénéiser l'écriture :  may the force be with you!
Découper une phrase avec espace (retourne une liste): ['LU3IN026', 'est', "l'UE", "d'IA", 'et', 'Sciences', 'des', 'données.']
Découper une phrase avec apostrophe (retourne une liste): ['LU3IN026 est l', 'UE d', 'IA et Sciences des données.']


In [40]:
# Avec le dataframe du dataset:
print("Premier message du train: ",df_spam['message'].iloc[0])

print("Résultat d'un découpage: ",df_spam['message'].iloc[0].split())

Premier message du train:  Go until jurong point  crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Résultat d'un découpage:  ['Go', 'until', 'jurong', 'point', 'crazy..', 'Available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet...', 'Cine', 'there', 'got', 'amore', 'wat...']


 Certains éléments d'une phrase ne sont pas utiles pour le traitement, par exemple en anglais, on peut utiliser la liste des mots inutiles suivants:

In [41]:
mots_inutiles = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `nettoyage` qui prend une chaîne de caractères et rend la chaîne après nettoyage: 1) mise en minuscules 2) remplacement des caractères de ponctuation par un espace (SAUF l'apostrophe qui doit rester car elle va être utilisée pour les mots inutiles).


In [45]:
def nettoyage(s):
    s = s.lower()
    ponctuation = string.punctuation.replace("'", "")
    for char in ponctuation:
        s = s.replace(char, " ")
    #return " ".join(s.split()) normalisation des espaces
    return s

In [46]:
# Exemple de nettoyage sur un texte simple: vérifiez ce qui est enlevé
nettoyage("LU3IN026 est l'UE d'IA et Sciences des données, mais il y a une autre UE d'IA en licence...")

"lu3in026 est l'ue d'ia et sciences des données  mais il y a une autre ue d'ia en licence   "

In [47]:
# Nettoyage du premier texte de notre corpus
nettoyage(df_spam['message'].iloc[0])

'go until jurong point  crazy   available only in bugis n great world la e buffet    cine there got amore wat   '

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `text2vect` qui prend une chaîne de caractères ainsi qu'une liste de mots inutiles et rend la liste composée par les mots de cette chaîne obtenus, après son nettoyage et après avoir enlevé les mots inutiles.



In [58]:
def text2vect(s, mots_inutiles):
    s = nettoyage(s)
    s_tokens = s.split()
    vect = [word for word in s_tokens if word not in mots_inutiles]
    
    return vect

In [59]:
text2vect("LU3IN026 est l'UE d'IA et Sciences des données, mais il y a une autre UE d'IA en licence...",mots_inutiles)

['lu3in026',
 'est',
 "l'ue",
 "d'ia",
 'et',
 'sciences',
 'des',
 'données',
 'mais',
 'il',
 'une',
 'autre',
 'ue',
 "d'ia",
 'en',
 'licence']

In [60]:
text2vect("May the Force be with you!",mots_inutiles)

['may', 'force']

In [61]:
text2vect("You shan't pass!",mots_inutiles)

['pass']

In [62]:
text2vect(df_spam['message'].iloc[0],mots_inutiles)

['go',
 'jurong',
 'point',
 'crazy',
 'available',
 'bugis',
 'n',
 'great',
 'world',
 'la',
 'e',
 'buffet',
 'cine',
 'got',
 'amore',
 'wat']

<font color="RED" size="+1"><b>[Q]</b></font> Ajouter une nouvelle colonne de nom `les_mots` au dataframe `df_spam` pour laquelle chaque ligne contient le résultat de l'application de `text2vect` sur le message de l'exemple correspondant.


In [65]:
df_spam['les_mots'] = df_spam['message'].apply(lambda x: text2vect(x, mots_inutiles))

In [66]:
print("Voilà le résultat attendu :")
df_spam

Voilà le résultat attendu :


,label,message,les_mots
0,ham,Go until jurong point crazy.. Available only ...,"[go, jurong, point, crazy, available, bugis, n..."
1,ham,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,ham,U dun say so early hor... U c already then say...,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,Nah I don't think he goes to usf he lives aro...,"[nah, think, goes, usf, lives, around, though]"
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"[2nd, time, tried, 2, contact, u, u, å£750, po..."
5568,ham,Will Ì_ b going to esplanade fr home?,"[ì, b, going, esplanade, fr, home]"
5569,ham,Pity * was in mood for that. So...any other s...,"[pity, mood, suggestions]"
5570,ham,The guy did some bitching but I acted like i'd...,"[guy, bitching, acted, like, i'd, interested, ..."


### Découpage du dataset en 2 ensembles train et test

<font color="RED" size="+1"><b>[Q]</b></font> Pour mettre au point nos fonctions dans ce TME, on va travailler sur une partie du dataset, par exemple, on peut prendre 5% des exemples (en respectant la distribution des classes, donc en prenant 5% d'exemples de chaque label).
Compléter le code ci-dessous pour définir les 2 variables `df_train` et `df_test` qui contiendront chacune 1 dataframe, partie de `df_spam`. Pour construire `df_train` on prend aléatoirement 5% des exemples de chaque classe, comme on l'a déjà fait dans un TME précédent.


In [67]:
np.random.seed(42)

# pourcentage d'exemples de chaque classe à garder:
taux = 0.05    # ici on prend 5% 

# déclaration des variables qui seront initialisées dans la boucle:
df_train = None  
df_test = None

all_train_ids = []
all_test_ids = []

for l in les_labels:
    nb_total = df_spam['label'].value_counts()[l]
    nb_pris = int(nb_total*taux) 
    print(f"Nombre d'exemples du label {l} pris pour apprendre: {nb_pris}")

    les_ids = df_spam[df_spam['label']==l].index.to_list()
    np.random.shuffle(les_ids)

    # ################################# COMPLETER ICI 

    all_train_ids.extend(les_ids[:nb_pris])
    all_test_ids.extend(les_ids[nb_pris:])

df_train = df_spam.loc[all_train_ids]
df_test = df_spam.loc[all_test_ids] 
    # ###########################################################
    
# Résultat:
print(f"Dimension de df_train:\t{df_train.shape}")
print(f"Dimension de df_test:\t{df_test.shape}")

Nombre d'exemples du label ham pris pour apprendre: 241
Nombre d'exemples du label spam pris pour apprendre: 37
Dimension de df_train:	(278, 3)
Dimension de df_test:	(5294, 3)


In [68]:
print("Exemple d'ensemble d'apprentissage : ")
df_train

Exemple d'ensemble d'apprentissage : 


,label,message,les_mots
3714,ham,I am late so call you tomorrow morning.take ca...,"[late, call, tomorrow, morning, take, care, sw..."
1311,ham,U r too much close to my heart. If u go away i...,"[u, r, much, close, heart, u, go, away, shatte..."
548,ham,Wait &lt;#&gt; min..,"[wait, lt, gt, min]"
1324,ham,Can you call me plz. Your number shows out of ...,"[call, plz, number, shows, coveragd, area, urg..."
3184,ham,MAYBE IF YOU WOKE UP BEFORE FUCKING 3 THIS WOU...,"[maybe, woke, fucking, 3, problem]"
...,...,...,...
1852,spam,This is the 2nd time we have tried 2 contact u...,"[2nd, time, tried, 2, contact, u, u, 750, poun..."
2582,spam,3 FREE TAROT TEXTS! Find out about your love l...,"[3, free, tarot, texts, find, love, life, try,..."
3418,spam,Do you want a new Video phone? 600 anytime any...,"[want, new, video, phone, 600, anytime, networ..."
3422,spam,Had your mobile 10 mths? Update to latest Oran...,"[mobile, 10, mths, update, latest, orange, cam..."


In [69]:
print("Exemple d'ensemble de test : ")
df_test

Exemple d'ensemble de test : 


,label,message,les_mots
3232,ham,Height of recycling: Read twice- People spend ...,"[height, recycling, read, twice, people, spend..."
4808,ham,Don't worry though I understand how important...,"[worry, though, understand, important, put, pl..."
2994,ham,Mm not entirely sure i understood that text bu...,"[mm, entirely, sure, understood, text, hey, ho..."
567,ham,So anyways you can just go to your gym or wha...,"[anyways, go, gym, whatever, love, smiles, hop..."
740,ham,Yes i will be there. Glad you made it.,"[yes, glad, made]"
...,...,...,...
2908,spam,URGENT! Your Mobile number has been awarded wi...,"[urgent, mobile, number, awarded, å£2000, priz..."
2704,spam,FreeMsg: Fancy a flirt? Reply DATE now & join ...,"[freemsg, fancy, flirt, reply, date, join, uks..."
3979,spam,ringtoneking 84484,"[ringtoneking, 84484]"
5074,spam,This is the 2nd attempt to contract U you hav...,"[2nd, attempt, contract, u, weeks, top, prize,..."


## Calculs de probabilités

À partir d'ici on va travailler sur les données d'apprentissage (`df_train`) pour déterminer les probabilités qui serviront à classer les messages.

<font color="RED" size="+1"><b>[Q]</b></font> Construire la liste de tous les mots présents dans tous les vecteurs de `df_train`, chaque mots doit n'apparaître qu'une seule fois dans la liste obtenue. Cette liste sera stockée une fois triée dans la variable `index_mots` (et on l'appelle par la suite "index de mots").


In [81]:
index_mots = set()
for mot in df_train['les_mots']:
    index_mots.update(mot) 

index_mots = sorted(list(index_mots))


In [82]:
print("Nombre de mots trouvés: ", len(index_mots))

print("Les 10 premiers mots trouvés :")
for i in range(0,10):
    print(f"\ten position {i} --> {index_mots[i]}")

print("pour contrôler, on regarde en différentes positions :")
for i in range(200,len(index_mots),250):
    print(f"\ten position {i} --> {index_mots[i]}")


Nombre de mots trouvés:  1377
Les 10 premiers mots trouvés :
	en position 0 --> '
	en position 1 --> 't
	en position 2 --> 0
	en position 3 --> 00
	en position 4 --> 01223585334
	en position 5 --> 0207
	en position 6 --> 03
	en position 7 --> 07801543489
	en position 8 --> 0800
	en position 9 --> 08000930705
pour contrôler, on regarde en différentes positions :
	en position 200 --> bcoz
	en position 450 --> everything
	en position 700 --> lift
	en position 950 --> project
	en position 1200 --> timings


Chaque message va maintenant être représenté comme un vecteur de valeurs 0 ou 1: ce vecteur possède autant de colonnes qu'il y a de mots dans `index_mots`. Ce vecteur est construit ainsi: pour un exemple $i$, la valeur de la colonne $j$ du vecteur vaudra 1 si la liste de mots de l'exemple $i$ contient le mot en position $j$ dans `index_mots`, ou 0 dans le cas contraire.

*Remarque:* même si un mot apparaît plusieurs fois dans la liste, cela ne compte que pour 1.

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `df2array` qui prend un dataframe `df` contenant la colonne `les_mots` ainsi qu'un index de mots et rend le `np.array` correspondant aux vecteurs de valeurs représentant les exemples de `df`. Les mots de `les_mots` qui ne sont pas dans l'index de mots ne sont pas pris en compte.  


In [103]:
def df2array(df, index_mots):
    matrice = np.zeros((len(df), len(index_mots)), dtype=int)
    mot_au_id = {mot: i for i, mot in enumerate(index_mots)}
    for i, message_words in enumerate(df['les_mots']):
        for mot in message_words:
            if mot in mot_au_id:
                matrice[i][mot_au_id[mot]] = 1   
    return matrice


In [ ]:
mat_train = df2array(df_train,index_mots)

print(f"Dimensions de la matrice obtenue : {mat_train.shape}")
#print(mat_train)

Dimensions de la matrice obtenue : (278, 1377)
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 0 0]]


In [105]:
print("Message 0:", df_train['les_mots'].iloc[0])

print("\nPositions non nulles dans le vecteur d'index:")

for i in range(0, len(mat_train[0])):
    if mat_train[0][i] == 1:
        print("\tcolonne ",i," : ", index_mots[i])
    

Message 0: ['late', 'call', 'tomorrow', 'morning', 'take', 'care', 'sweet', 'dreams', 'u', 'ummifying', 'bye']

Positions non nulles dans le vecteur d'index:
	colonne  245  :  bye
	colonne  248  :  call
	colonne  265  :  care
	colonne  412  :  dreams
	colonne  679  :  late
	colonne  791  :  morning
	colonne  1150  :  sweet
	colonne  1154  :  take
	colonne  1209  :  tomorrow
	colonne  1238  :  u
	colonne  1244  :  ummifying


In [107]:
# Compléter la ligne suivante pour afficher le nombre de lignes pour les 2 labels
for e in les_labels:
    count = df_train[df_train['label'] == e].shape[0]
    print(f"Il y a {count} exemples de la classe {e}")

Il y a 241 exemples de la classe ham
Il y a 37 exemples de la classe spam


In [28]:
# Résultat attendu :

Il y a 241 exemples de la classe ham
Il y a 37 exemples de la classe spam


<font color="RED" size="+1"><b>[Q]</b></font> Construire un dictionnaire qui donne, pour chaque label $l$, pour chaque mot $m$ de l'index des mots, la fréquence de $m$ parmi les exemples de `df_train` qui ont le label $l$.


*Remarque*: penser à une solution dans boucle for sur les exemples...


In [113]:
frequences = dict()

################### A COMPLETER 

for e in les_labels: 
    df_label = df_train[df_train['label'] == e]
    mat = df2array(df_label, index_mots)
    mat_sum = mat.sum(axis=0)
    nb_exemples = mat.shape[0]
    freq_values = mat_sum / nb_exemples
    frequences[e] = list(freq_values)


In [114]:
# Affichage de quelques valeurs de fréquence non nulles 
print("Seuls les 10 premiers non nuls sont affichés.")        
for l in frequences:
    nb= 0
    print("Pour le label",l, ":")
    for j in range(0,len(frequences[l])):
        if frequences[l][j] != 0:
            if (nb < 10):
                print(f'\t {index_mots[j]}:\t {frequences[l][j]:0.6f}')
            nb +=1


Seuls les 10 premiers non nuls sont affichés.
Pour le label ham :
	 ':	 0.004149
	 't:	 0.004149
	 1:	 0.016598
	 100:	 0.004149
	 1st:	 0.004149
	 2:	 0.058091
	 2marrow:	 0.004149
	 2nd:	 0.004149
	 3:	 0.020747
	 30:	 0.004149
Pour le label spam :
	 0:	 0.027027
	 00:	 0.054054
	 01223585334:	 0.027027
	 0207:	 0.027027
	 03:	 0.027027
	 07801543489:	 0.027027
	 0800:	 0.027027
	 08000930705:	 0.054054
	 08000938767:	 0.027027
	 08002888812:	 0.027027


In [115]:
print("Frequences max:")
for e in les_labels:
    print(f"\tpour ham: {max(frequences[e]):1.4f} pour le mot '{index_mots[np.argmax(frequences[e])]}'")


Frequences max:
	pour ham: 0.1867 pour le mot 'u'
	pour ham: 0.4865 pour le mot 'call'


## Classification de textes

<div class="alert alert-block alert-warning">
On considère deux variables $X$ et $Y$ :
    <ul>
        <li>$X$ est un message</li>
        <li>$Y$ est le label d'un message et peut prendre 2 valeurs : <code>'ham'</code> et <code>'spam'</code></li>
    </ul>

Avec les fonctions précédentes on peut représenter les messages d'un corpus de documents sous la forme d'un vecteur $X \in \{0, 1\}^p$ où $p$ est le nombre total de mots de l'index. Le $i$ème message est représenté par le vecteur ${\bf x}_i = [x_{i1} \dots x_{ip}]$, où $x_{ij}$ vaut 1 si le $j$ème mot de l'index est présent dans le message $i$, et 0 sinon.
    
Comme vu en cours, pour un classifieur bayésien, nous devons estimer, à partir de la base <code>df_train</code>, les probabilités $p(ham)$, $p(spam)$, $p(X |ham)$ et $p(X | spam)$.

Les 2 premières sont simples à calculer : on compte le nombre de fois où le label apparaît parmi les exemples.

Pour un label $Y$ et un exemple ${\bf x}$, le calcul de $p({\bf x} | Y)$ se fait en utilisant les probabilités $p(mot | Y)$ de tous les mots qui composent ${\bf x}$ (et qui sont des mots de l'index des mots). $p(mot | Y)$ est la probabilité que le mot <code>mot</code> apparaisse dans un message sachant que ce message appartient à la classe $Y$.
    
On pose ainsi

$$ p({\bf x} | Y) = \prod_{mot \in index\_mots} p(mot | Y)^{x_{mot}} \left(1 - p(mot | Y)\right)^{1 - x_{mot}} $$

où $x_{mot}$ correspond à $1$ ou $0$ selon que le `mot` apparaît dans ${\bf x}$ ou pas. Ce terme permet de retenir soit la probabilité $p(mot | Y)$ si le mot est dans ${\bf x}$, soit la probabilité qu'il n'y soit pas ($1-p(mot |Y)$).
    
Une fois que $p({\bf x} | Y)$ est calculée, on peut estimer $p(Y|X)$ grâce au théorème de Bayes (cf. cours 6):

$$p(Y|{\bf x}) = p(Y) p({\bf x} | Y)$$

avec $Y$ qui vaut soit 'ham', soit 'spam'.    

    
Une fois $p(Y{{\bf x}})$ calculée pour chaque valeur de label, pour prédire le label de ${\bf x}$ on choisit le label qui possède la plus forte probabilité.
</div>

In [116]:
# calculer p(ham) et p(spam) dans df_spam:

# ------------------------ (CORRECTION POUR ENSEIGNANT)
for l in les_labels:
    # #################### A COMPLETER 
    df_l = df_train[df_train['label'] == l] 
    proba = df_l.shape[0] / df_train.shape[0]
    ##########################
    print(f'p({l:4}) = {proba:0.4f}')


p(ham ) = 0.8669
p(spam) = 0.1331


<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `proba_mot` qui étant donné un mot, un label, une liste de mots, et un dictionnaire avec les fréquences des mots par label (comme `frequences` et compatible avec la liste de mots) rend $p(mot| label)$ la probabilité du mot d'appartenir au label donné.


In [130]:
def proba_mot(mot, label, index_mots, frequences):
    if label not in frequences:
        return 0.0
    if mot in index_mots:
        idx = index_mots.index(mot)
        return frequences[label][idx]
    return 0.0

In [131]:
# probabilité d'un mot qui n'est pas dans l'index:
print(f"{proba_mot("toto", "ham", index_mots, frequences)}")

0.0


In [132]:
# probabilité d'un mot de l'index pour un label qui n'existe pas:
print(f"{proba_mot("visit", "cookie", index_mots, frequences)}")

0.0


In [133]:
# probabilité d'un mot de l'index pour un label qui existe:
print(f"{proba_mot("call", "ham", index_mots, frequences)}")

0.05394190871369295


In [134]:
# probabilité d'un mot de l'index pour un label qui existe:
print(f"{proba_mot("call", "spam", index_mots, frequences)}")

0.4864864864864865


<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `proba_exemple` qui étant donné un exemple représenté sous la forme d'une liste de mots, un label, une liste de mots, et un dictionnaire comme `frequences` (compatible avec la liste de mots) rend $p(exemple|label)$ la probabilité de l'exemple d'appartenir au label.


In [135]:
def proba_exemple(message, label, index_mots,frequences): 
    result = 1
    for m in index_mots: 
        p = proba_mot(m, label, index_mots, frequences)
        x = 0
        if m in message: 
            x = 1
        result *= ((p**x) * ((1 - p)**(1-x)))
    return result

In [136]:
print(f"exemple de proba: {proba_exemple(df_train['les_mots'].iloc[10], 'ham', index_mots,frequences):1.5e}")

exemple de proba: 1.74347e-16


In [137]:
p = dict()
for i in range(0,len(df_train)):
    for e in les_labels:
        p[e] = proba_exemple(df_train['les_mots'].iloc[i], e, index_mots,frequences)
    #p_ham = proba_exemple(df_train['les_mots'].iloc[i], 'ham', index_mots,frequences)
    if (p['spam']>0) and (p['ham']>0):
        print(f"Exemple {i} : p(ham)= {p['ham']:1.5e}\tp(spam)= {p['spam']:1.5e}")


Exemple 155 : p(ham)= 3.44444e-04	p(spam)= 5.69181e-09
Exemple 230 : p(ham)= 6.21134e-10	p(spam)= 2.32295e-12


<font color="RED" size="+1"><b>[Q]</b></font> À partir de `mat_train` et de `frequences` et en utilisant des opérations matricielles, on peut aussi calculer les probabilités pour chaque label de tous les exemples. En terme de temps de calcul, cela peut être plus efficace.

Donner l'instruction permettant de calculer toutes ces probabilités, puis vérifier que vous trouver les mêmes valeurs pour les exemples précédents.


In [139]:
toutes_probas = dict()
for l in les_labels:
    p_mot_y = np.array(frequences[l]) 
    prob_matrix = mat_train * p_mot_y + (1 - mat_train) * (1 - p_mot_y)
    toutes_probas[l] = np.prod(prob_matrix, axis=1)


In [140]:
p = dict()
for i in range(0,len(df_train)):
    for e in les_labels:
        p[e] = toutes_probas[e][i]
    if (p['spam']>0) and (p['ham']>0):
        print(f"Exemple {i} : p(ham)= {p['ham']:1.5e}\tp(spam)= {p['spam']:1.5e}")



Exemple 155 : p(ham)= 3.44444e-04	p(spam)= 5.69181e-09
Exemple 230 : p(ham)= 6.21134e-10	p(spam)= 2.32295e-12


<font color="RED" size="+1"><b>[Q]</b></font> En utilisant les fonctions écrites, calculer le taux de bonne prédiction du classifieur bayésien naïf pour chaque valeur de label pour le dataset d'apprentissage. 

*Remarque*: une solution efficace peut ne pas utiliser toutes les fonctions précédentes...

In [ ]:
# La variable suivante permettra de stocker les résultats
taux = dict()
toutes_probas = dict()
for l in les_labels:
    taux[l] = dict()
    taux[l][True] = 0    # nombre de bien classés
    taux[l][False] = 0   # nombre de mal classés
    toutes_probas[l] = np.zeros(1)
tic = time.time()

df = df_train
mat_df = df2array(df,index_mots)

############### A COMPLETER ############
p_y = {l: len(df[df['label'] == l]) / len(df) for l in les_labels}
for l in les_labels:
    p_mot_y = np.array(frequences[l])
    prob_matrix = mat_df * p_mot_y + (1 - mat_df) * (1 - p_mot_y)
    toutes_probas[l] = p_y[l] * np.prod(prob_matrix, axis=1)

y_true = df['label'].values

for i in range(len(df)):
    if toutes_probas['ham'][i] >= toutes_probas['spam'][i]:
        pred = 'ham'
    else:
        pred = 'spam'
    
    label_reel = y_true[i]
    taux[label_reel][pred == label_reel] += 1
    
for l in les_labels:
    total = taux[l][True] + taux[l][False]
    accuracy = taux[l][True] / total if total > 0 else 0
    print(f"Taux de bonne classification pour {l}: {accuracy:.2%}")

########################################

toc = time.time()
print("Résultats des classements : ", taux)

print(f"Durée du calcul : {(toc-tic):1.3f} secondes.")
# le résultat total peut prendre un certain temps...


# Il reste à calculer les taux de bonne classification par label

Taux de bonne classification для ham: 100.00%
Taux de bonne classification для spam: 100.00%
Résultats des classements :  {'ham': {True: 241, False: 0}, 'spam': {True: 37, False: 0}}
Durée du calcul : 0.007 secondes.


<font color="RED" size="+1"><b>[Q]</b></font> Même question pour calculer le taux de bonne classification pour le dataset de test.

*Remarque*: L'index des mots ainsi que la matrice des fréquences restent les mêmes, ce sont toujours celles construites à partir de la base d'apprentissage. Par contre, la matrice des présences doit, elle, être recalculées.

In [143]:
# La variable suivante permettra de stocker les résultats
taux = dict()
toutes_probas = dict()
for l in les_labels:
    taux[l] = dict()
    taux[l][True] = 0    # nombre de bien classés
    taux[l][False] = 0   # nombre de mal classés
    toutes_probas[l] = np.zeros(1)
tic = time.time()

df = df_test
mat_df = df2array(df,index_mots)

############### A COMPLETER ############

p_y = {l: len(df[df['label'] == l]) / len(df) for l in les_labels}
for l in les_labels:
    p_mot_y = np.array(frequences[l])
    prob_matrix = mat_df * p_mot_y + (1 - mat_df) * (1 - p_mot_y)
    toutes_probas[l] = p_y[l] * np.prod(prob_matrix, axis=1)

y_true = df['label'].values

for i in range(len(df)):
    if toutes_probas['ham'][i] >= toutes_probas['spam'][i]:
        pred = 'ham'
    else:
        pred = 'spam'
    
    label_reel = y_true[i]
    taux[label_reel][pred == label_reel] += 1
    
for l in les_labels:
    total = taux[l][True] + taux[l][False]
    accuracy = taux[l][True] / total if total > 0 else 0
    print(f"Taux de bonne classification для {l}: {accuracy:.2%}")



########################################
toc = time.time()
print("Résultats des classements : ", taux)

print(f"Durée du calcul : {(toc-tic):1.3f} secondes.")
# le résultat total peut prendre un certain temps...


# Il reste à calculer les taux de bonne classification par label

Taux de bonne classification для ham: 99.30%
Taux de bonne classification для spam: 27.46%
Résultats des classements :  {'ham': {True: 4552, False: 32}, 'spam': {True: 195, False: 515}}
Durée du calcul : 0.115 secondes.


## Evaluation du classifieur Naive Bayes

Pour tout ce que l'on a fait jusque-là, on a travaillé sur tout le dataset, pour pouvoir l'évaluer il est nécessaire de le séparer en ensemble d'apprentissage et ensemble de test.

<font color="RED" size="+1"><b>[Q]</b></font> Découper aléatoirement `df_spam` en 2 parties égales, chacune contenant des exemples des 2 labels, avec la même distribution des labels dans chaque partie. 
Une des parties sert à apprendre l'index des mots et leurs fréquences.
L'autre partie n'est utilisée que pour calculer le taux de bonne classification par label.

Donner ensuite les taux de bonne classification par label pour l'ensemble de train et pour l'ensemble de test.

*Remarque*: certains mots de la partie de test pourront ne pas être présents dans l'index de mots car ils peuvent être absents de la partie d'apprentissage.

In [150]:
def eval_model(df, frequences, index_mots):
    res_taux = {l: {True: 0, False: 0} for l in les_labels}
    toutes_probas = {}
    mat_df = df2array(df, index_mots)
    p_y = {l: len(df[df['label'] == l]) / len(df) for l in les_labels}
    for l in les_labels:
        p_mot_y = np.array(frequences[l])
        prob_matrix = mat_df * p_mot_y + (1 - mat_df) * (1 - p_mot_y)
        toutes_probas[l] = p_y[l] * np.prod(prob_matrix + 1e-100, axis=1)

    y_true = df['label'].values
    for i in range(len(df)):
        pred = 'ham' if toutes_probas['ham'][i] >= toutes_probas['spam'][i] else 'spam'
        
        label_reel = y_true[i]
        res_taux[label_reel][pred == label_reel] += 1

    for l in les_labels:
        total = res_taux[l][True] + res_taux[l][False]
        accuracy = res_taux[l][True] / total if total > 0 else 0
        print(f"Taux de bonne classification pour {l}: {accuracy:.2%}")

    return res_taux



In [151]:
np.random.seed(42)

# pourcentage d'exemples de chaque classe à garder:
taux = 0.5    # ici on prend 5% 

# déclaration des variables qui seront initialisées dans la boucle:
df_train = None  
df_test = None

all_train_ids = []
all_test_ids = []

for l in les_labels:
    nb_total = df_spam['label'].value_counts()[l]
    nb_pris = int(nb_total*taux) 
    print(f"Nombre d'exemples du label {l} pris pour apprendre: {nb_pris}")

    les_ids = df_spam[df_spam['label']==l].index.to_list()
    np.random.shuffle(les_ids)

    # ################################# COMPLETER ICI 

    all_train_ids.extend(les_ids[:nb_pris])
    all_test_ids.extend(les_ids[nb_pris:])

df_train = df_spam.loc[all_train_ids]
df_test = df_spam.loc[all_test_ids] 
    # ###########################################################
    
# Résultat:
print(f"Dimension de df_train:\t{df_train.shape}")
print(f"Dimension de df_test:\t{df_test.shape}")

index_mots_train = sorted(list({m for msg in df_train['les_mots'] for m in msg}))
frequences_train = {}
for l in les_labels:
    df_label = df_train[df_train['label'] == l]
    mat = df2array(df_label, index_mots_train)
    frequences_train[l] = mat.sum(axis=0) / mat.shape[0]

eval_model(df_train, frequences_train, index_mots_train)
eval_model(df_test, frequences_train, index_mots_train)


Nombre d'exemples du label ham pris pour apprendre: 2412
Nombre d'exemples du label spam pris pour apprendre: 373
Dimension de df_train:	(2785, 3)
Dimension de df_test:	(2787, 3)
Taux de bonne classification pour ham: 100.00%
Taux de bonne classification pour spam: 99.73%
Taux de bonne classification pour ham: 99.88%
Taux de bonne classification pour spam: 83.69%


{'ham': {True: 2410, False: 3}, 'spam': {True: 313, False: 61}}

<font color="RED" size="+1"><b>[Q]</b></font> Mettre en place une approche par validation croisée pour évaluer le taux de bonne classification moyen de cette approche. 

In [152]:
K = 5
df_shuffled = df_spam.sample(frac=1, random_state=42).reset_index(drop=True)
folds = np.array_split(df_shuffled.index.to_list(), K)
final_scores = {l: [] for l in les_labels}
for i in range(K):
    test_ids = folds[i]
    train_ids = np.concatenate([folds[j] for j in range(K) if j != i])

    df_train_fold = df_shuffled.loc[train_ids]
    df_test_fold = df_shuffled.loc[test_ids]
    
    index_mots_fold = sorted(list({m for msg in df_train_fold['les_mots'] for m in msg}))
    
    frequences_fold = {}
    for l in les_labels:
        df_label = df_train_fold[df_train_fold['label'] == l]
        mat = df2array(df_label, index_mots_fold)
        frequences_fold[l] = mat.sum(axis=0) / mat.shape[0]
    
    res_step = eval_model(df_test_fold, frequences_fold, index_mots_fold)
    
    for l in les_labels:
        total = res_step[l][True] + res_step[l][False]
        acc = res_step[l][True] / total if total > 0 else 0
        final_scores[l].append(acc)
    

for l in les_labels:
    mean_acc = np.mean(final_scores[l])
    std_acc = np.std(final_scores[l]) 
    print(f"Label {l}: {mean_acc:.2%} (+/- {std_acc:.2%})")

Taux de bonne classification pour ham: 99.48%
Taux de bonne classification pour spam: 86.67%
Taux de bonne classification pour ham: 99.79%
Taux de bonne classification pour spam: 88.59%
Taux de bonne classification pour ham: 99.90%
Taux de bonne classification pour spam: 87.07%
Taux de bonne classification pour ham: 99.59%
Taux de bonne classification pour spam: 88.19%
Taux de bonne classification pour ham: 100.00%
Taux de bonne classification pour spam: 85.99%
Label ham: 99.75% (+/- 0.19%)
Label spam: 87.30% (+/- 0.96%)


<font color="RED" size="+1"><b>[Q]</b></font> Comparer les résultats obtenus (taux de bonne classification avec la validation croisée, temps de calcul) avec ceux obtenus avec l'application d'un classifieur par $k$ plus proches voisins. Pour cela, la `mat_spam` doit être utilisée comme description des données. 